# SQL Database Ingestion for RAG

This notebook demonstrates ingestion from SQLite with clean schema, query extraction, metadata, and chunking.

Scope:
- Create/seed sample DB if missing
- Extract rows into Documents
- Prepare chunks for retrieval

In [ ]:
import sqlite3
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "company.db"

DATA_DIR.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        role TEXT,
        department TEXT,
        salary REAL
    )
    """
)

cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS projects (
        id INTEGER PRIMARY KEY,
        name TEXT,
        status TEXT,
        budget REAL,
        lead_id INTEGER
    )
    """
)

conn.commit()
print("DB ready at:", DB_PATH)

In [ ]:
employees = [
    (1, "John Doe", "Senior Developer", "Engineering", 95000),
    (2, "Jane Smith", "Data Scientist", "Analytics", 105000),
    (3, "Mike Johnson", "Product Manager", "Product", 110000),
]

projects = [
    (1, "RAG Implementation", "Active", 150000, 1),
    (2, "Data Pipeline", "Completed", 80000, 2),
    (3, "Customer Portal", "Planning", 200000, 3),
]

cursor.executemany("INSERT OR REPLACE INTO employees VALUES (?,?,?,?,?)", employees)
cursor.executemany("INSERT OR REPLACE INTO projects VALUES (?,?,?,?,?)", projects)
conn.commit()

print("Seeded employees:", len(employees))
print("Seeded projects:", len(projects))

In [ ]:
cursor.execute("""
SELECT e.name, e.role, e.department, e.salary, p.name, p.status, p.budget
FROM employees e
LEFT JOIN projects p ON e.id = p.lead_id
ORDER BY e.id
""")
rows = cursor.fetchall()

sql_docs = []
for idx, row in enumerate(rows, start=1):
    emp_name, role, dept, salary, project_name, project_status, project_budget = row
    content = (
        f"Employee: {emp_name}\n"
        f"Role: {role}\n"
        f"Department: {dept}\n"
        f"Salary: {salary}\n"
        f"Project: {project_name}\n"
        f"Project Status: {project_status}\n"
        f"Project Budget: {project_budget}"
    )
    sql_docs.append(
        Document(
            page_content=content,
            metadata={
                "source": str(DB_PATH),
                "record_type": "employee_project_join",
                "row_number": idx,
                "employee_name": emp_name,
            },
        )
    )

print("Extracted docs:", len(sql_docs))
print("First doc metadata:", sql_docs[0].metadata)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=70,
)
sql_chunks = splitter.split_documents(sql_docs)

print("Chunks:", len(sql_chunks))
print("First chunk:")
print(sql_chunks[0].page_content)

conn.close()